In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
import os
from tqdm import tqdm
import s3fs
import joblib
from scipy import stats

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

## Helper Classes

In [ ]:
class ModelComparison:
    """Compare SARIMA and Prophet models."""
    
    def __init__(self, str_bucket, str_dirname_output):
        self.str_bucket = str_bucket
        self.str_dirname_output = str_dirname_output
        self.s3_client = boto3.client('s3')
        self.df_train = None
        self.df_test = None
        self.sarima_model = None
        self.prophet_model = None
        self.df_forecast_sarima = None
        self.df_forecast_prophet = None
        self.dict_metrics_sarima = {}
        self.dict_metrics_prophet = {}
    
    def import_splits(self):
        """Load train/test splits."""
        str_train_uri = f's3://{self.str_bucket}/02_preprocessing/train_data.csv'
        str_test_uri = f's3://{self.str_bucket}/02_preprocessing/test_data.csv'
        
        self.df_train = pd.read_csv(str_train_uri)
        self.df_test = pd.read_csv(str_test_uri)
        
        self.df_train['date'] = pd.to_datetime(self.df_train['date'])
        self.df_test['date'] = pd.to_datetime(self.df_test['date'])
        
        print(f'Loaded splits: {len(self.df_train)} train, {len(self.df_test)} test')
        return self.df_train, self.df_test
    
    def load_models(self):
        """Load serialized SARIMA and Prophet models."""
        try:
            # Download from S3
            self.s3_client.download_file(
                self.str_bucket,
                '03_sarima/sarima_model.pkl',
                './sarima_model.pkl'
            )
            self.sarima_model = joblib.load('./sarima_model.pkl')
            print('Loaded SARIMA model')
        except Exception as e:
            print(f'Could not load SARIMA model: {e}')
        
        try:
            self.s3_client.download_file(
                self.str_bucket,
                '04_prophet/prophet_model.pkl',
                './prophet_model.pkl'
            )
            self.prophet_model = joblib.load('./prophet_model.pkl')
            print('Loaded Prophet model')
        except Exception as e:
            print(f'Could not load Prophet model: {e}')
    
    def generate_forecasts(self):
        """Generate forecasts from both models on test set."""
        # SARIMA forecast
        if self.sarima_model:
            forecast_result = self.sarima_model.get_forecast(steps=len(self.df_test))
            df_ci = forecast_result.conf_int(alpha=0.05)
            self.df_forecast_sarima = df_ci.copy()
            self.df_forecast_sarima.columns = ['ci_lower', 'ci_upper']
            self.df_forecast_sarima['forecast'] = forecast_result.predicted_mean.values
            self.df_forecast_sarima['model'] = 'SARIMA'
            print('Generated SARIMA forecast')
        
        # Prophet forecast
        if self.prophet_model:
            df_future = self.prophet_model.make_future_dataframe(periods=len(self.df_test), freq='MS')
            forecast = self.prophet_model.predict(df_future)
            df_forecast_future = forecast[forecast['ds'] > self.df_train['date'].max()][['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
            
            self.df_forecast_prophet = df_forecast_future.copy()
            self.df_forecast_prophet.columns = ['date', 'forecast', 'ci_lower', 'ci_upper']
            self.df_forecast_prophet['model'] = 'Prophet'
            print('Generated Prophet forecast')
    
    def calculate_metrics(self):
        """Calculate evaluation metrics for both models."""
        arr_actual = self.df_test['attrition_rate'].values
        
        if self.df_forecast_sarima is not None:
            arr_pred_sarima = self.df_forecast_sarima['forecast'].values[:len(arr_actual)]
            flt_rmse_sarima = np.sqrt(np.mean((arr_actual - arr_pred_sarima) ** 2))
            flt_mae_sarima = np.mean(np.abs(arr_actual - arr_pred_sarima))
            flt_mape_sarima = np.mean(np.abs((arr_actual - arr_pred_sarima) / arr_actual)) * 100
            
            self.dict_metrics_sarima = {
                'Model': 'SARIMA',
                'RMSE': flt_rmse_sarima,
                'MAE': flt_mae_sarima,
                'MAPE': flt_mape_sarima
            }
        
        if self.df_forecast_prophet is not None:
            arr_pred_prophet = self.df_forecast_prophet['forecast'].values[:len(arr_actual)]
            flt_rmse_prophet = np.sqrt(np.mean((arr_actual - arr_pred_prophet) ** 2))
            flt_mae_prophet = np.mean(np.abs(arr_actual - arr_pred_prophet))
            flt_mape_prophet = np.mean(np.abs((arr_actual - arr_pred_prophet) / arr_actual)) * 100
            
            self.dict_metrics_prophet = {
                'Model': 'Prophet',
                'RMSE': flt_rmse_prophet,
                'MAE': flt_mae_prophet,
                'MAPE': flt_mape_prophet
            }
    
    def plot_metrics_comparison(self):
        """Side-by-side metrics comparison."""
        df_comparison = pd.DataFrame([
            self.dict_metrics_sarima,
            self.dict_metrics_prophet
        ])
        
        print('\n=== METRICS COMPARISON ===')
        print(df_comparison.to_string(index=False))
        
        # Bar plot
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        metrics_list = ['RMSE', 'MAE', 'MAPE']
        colors = ['#1f77b4', '#ff7f0e']
        
        for int_idx, str_metric in enumerate(metrics_list):
            list_vals = [df_comparison[df_comparison['Model'] == 'SARIMA'][str_metric].values[0],
                         df_comparison[df_comparison['Model'] == 'Prophet'][str_metric].values[0]]
            axes[int_idx].bar(['SARIMA', 'Prophet'], list_vals, color=colors)
            axes[int_idx].set_title(str_metric, fontsize=12, fontweight='bold')
            axes[int_idx].set_ylabel(str_metric)
            axes[int_idx].grid(True, alpha=0.3, axis='y')
            
            # Add value labels
            for int_i, flt_val in enumerate(list_vals):
                axes[int_idx].text(int_i, flt_val, f'{flt_val:.4f}', ha='center', va='bottom')
        
        plt.tight_layout()
        str_path = f'{self.str_dirname_output}/16_metrics_comparison.png'
        plt.savefig(str_path, bbox_inches='tight', dpi=150)
        print(f'\nSaved metrics plot to {str_path}')
        plt.show()
        
        return df_comparison
    
    def plot_forecast_overlay(self):
        """Overlay forecast plots from both models."""
        fig, ax = plt.subplots(figsize=(14, 7))
        
        # Training data
        ax.plot(self.df_train['date'], self.df_train['attrition_rate'], linewidth=2, 
               label='Training Data', marker='o', markersize=4, color='#1f77b4')
        
        # Test data
        ax.plot(self.df_test['date'], self.df_test['attrition_rate'], linewidth=2.5, 
               label='Test Data (Actual)', marker='s', markersize=5, color='#2ca02c')
        
        # SARIMA forecast
        if self.df_forecast_sarima is not None:
            ax.plot(self.df_test['date'], self.df_forecast_sarima['forecast'].values[:len(self.df_test)], 
                   linewidth=2.5, label='SARIMA Forecast', marker='^', markersize=5, color='#ff7f0e')
        
        # Prophet forecast
        if self.df_forecast_prophet is not None:
            ax.plot(self.df_test['date'], self.df_forecast_prophet['forecast'].values[:len(self.df_test)], 
                   linewidth=2.5, label='Prophet Forecast', marker='d', markersize=5, color='#d62728')
        
        ax.set_xlabel('Date', fontsize=11)
        ax.set_ylabel('Attrition Rate', fontsize=11)
        ax.set_title('Model Forecast Comparison', fontsize=13, fontweight='bold')
        ax.legend(loc='best', fontsize=10)
        ax.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
        plt.tight_layout()
        
        str_path = f'{self.str_dirname_output}/17_forecast_overlay.png'
        plt.savefig(str_path, bbox_inches='tight', dpi=150)
        print(f'Saved overlay plot to {str_path}')
        plt.show()
    
    def plot_residuals_comparison(self):
        """Compare residuals from both models."""
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        arr_actual = self.df_test['attrition_rate'].values
        
        # SARIMA residuals
        if self.df_forecast_sarima is not None:
            arr_resid_sarima = arr_actual - self.df_forecast_sarima['forecast'].values[:len(arr_actual)]
            
            axes[0, 0].hist(arr_resid_sarima, bins=15, alpha=0.7, edgecolor='black', color='#ff7f0e')
            axes[0, 0].set_title('SARIMA Residual Distribution', fontsize=12, fontweight='bold')
            axes[0, 0].set_xlabel('Residual')
            axes[0, 0].grid(True, alpha=0.3, axis='y')
            
            axes[1, 0].plot(arr_resid_sarima, linewidth=1.5, marker='o', markersize=4, color='#ff7f0e')
            axes[1, 0].axhline(y=0, color='r', linestyle='--', alpha=0.5)
            axes[1, 0].set_title('SARIMA Residuals Over Time', fontsize=12, fontweight='bold')
            axes[1, 0].set_ylabel('Residual')
            axes[1, 0].grid(True, alpha=0.3)
        
        # Prophet residuals
        if self.df_forecast_prophet is not None:
            arr_resid_prophet = arr_actual - self.df_forecast_prophet['forecast'].values[:len(arr_actual)]
            
            axes[0, 1].hist(arr_resid_prophet, bins=15, alpha=0.7, edgecolor='black', color='#d62728')
            axes[0, 1].set_title('Prophet Residual Distribution', fontsize=12, fontweight='bold')
            axes[0, 1].set_xlabel('Residual')
            axes[0, 1].grid(True, alpha=0.3, axis='y')
            
            axes[1, 1].plot(arr_resid_prophet, linewidth=1.5, marker='o', markersize=4, color='#d62728')
            axes[1, 1].axhline(y=0, color='r', linestyle='--', alpha=0.5)
            axes[1, 1].set_title('Prophet Residuals Over Time', fontsize=12, fontweight='bold')
            axes[1, 1].set_ylabel('Residual')
            axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        str_path = f'{self.str_dirname_output}/18_residuals_comparison.png'
        plt.savefig(str_path, bbox_inches='tight', dpi=150)
        print(f'Saved residuals plot to {str_path}')
        plt.show()

## Constants

In [ ]:
str_bucket = 'time-series-forecasting-demo-repo'
str_task = 'employee_attrition_forecasting'
str_dirname_output = './output'

## Output Directory

In [ ]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

print(f'Output directory: {str_dirname_output}')

## Load Data and Models

In [ ]:
comparison = ModelComparison(str_bucket, str_dirname_output)
df_train, df_test = comparison.import_splits()
comparison.load_models()

## Generate Forecasts

In [ ]:
comparison.generate_forecasts()

## Calculate Metrics

In [ ]:
comparison.calculate_metrics()

## Metrics Comparison Table

In [ ]:
df_metrics = comparison.plot_metrics_comparison()

## Forecast Overlay Plot

In [ ]:
comparison.plot_forecast_overlay()

## Residuals Comparison

In [ ]:
comparison.plot_residuals_comparison()

## Final Comparison Summary

In [ ]:
print(f'\n=== FINAL MODEL COMPARISON ===')
print(f'\nDataset:')
print(f'  Train samples: {len(df_train)}')
print(f'  Test samples: {len(df_test)}')
print(f'\nModel Performance:')
print(df_metrics.to_string(index=False))

# Determine winner
if comparison.dict_metrics_sarima and comparison.dict_metrics_prophet:
    flt_mape_sarima = comparison.dict_metrics_sarima['MAPE']
    flt_mape_prophet = comparison.dict_metrics_prophet['MAPE']
    
    str_winner = 'SARIMA' if flt_mape_sarima < flt_mape_prophet else 'Prophet'
    flt_improvement = abs(flt_mape_sarima - flt_mape_prophet)
    
    print(f'\nWinner: {str_winner} (MAPE difference: {flt_improvement:.2f}%)')
    print(f'\nRecommendation:')
    if str_winner == 'SARIMA':
        print(f'  - SARIMA provides superior accuracy for production forecasting')
        print(f'  - Prophet offers better interpretability of trend/seasonality')
        print(f'  - Consider ensemble approach for robustness')
    else:
        print(f'  - Prophet provides superior accuracy and interpretability')
        print(f'  - Easier to integrate seasonality and trend changes')
        print(f'  - Better for stakeholder communication')

print(f'\n=== COMPARISON COMPLETE ===')